# Семинар 18: CTR prediction на KION

В этом семинаре мы строим полный **candidate generation -> reranking -> deep CTR -> sequential CTR** пайплайн на датасете KION. Базовый легкий candidate/rerank pipeline сделан по мотивам `Seminar5`, а deep/sequential часть достроена поверх него.

Что делаем:
1. Берем KION (`interactions/users/items`).
2. Генерируем кандидатов через `EASE`.
3. Собираем CTR-датасет для `train/val/test` stage-окнами.
4. Обучаем `LGBMClassifier` и `LGBMRanker`, сравниваем по `Recall@10 / NDCG@10`.
5. Обучаем `DCNv2 (lite)` и `FinalNet (lite)`.
6. Обучаем `TransAct-style (lite)` sequence model.
7. Сводим все модели в один лидерборд.

> Deep-модели ниже сделаны в компактном seminar-friendly формате, вдохновленном экосистемой **RecZoo / FuxiCTR / BARS / LongCTR**. Это удобно для учебного ноутбука и не требует внешнего репозитория внутри занятия.


## Источники архитектур

- `DCNv2`, `FinalNet`, `TransAct` перечислены в `reczoo/FuxiCTR`: https://github.com/reczoo/FuxiCTR
- benchmark-ориентированный репозиторий `reczoo/BARS`: https://github.com/reczoo/BARS
- long-sequence benchmark с `TransAct`: https://github.com/reczoo/LongCTR

См. также:
- в `FuxiCTR` есть ссылка на пример запуска моделей через BARS benchmark;
- в `LongCTR` `TransAct` указан как long-sequence CTR baseline.


In [18]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import torch

from ctr_seminar_utils import (
    DCNv2Lite,
    EASE,
    FeatureEncoder,
    FinalNetLite,
    SplitConfig,
    TransActLite,
    build_leaderboard,
    build_stage_dataset,
    evaluate_ranking,
    find_data_root,
    fit_lgbm_models,
    load_kion_data,
    make_stage_windows,
    predict_torch_model,
    prepare_lgbm_matrices,
    preprocess_kion,
    set_seed,
    train_torch_model,
)

warnings.filterwarnings('ignore')
set_seed(42)
pd.set_option('display.max_columns', 200)


## 1. Конфиг

Если ноутбук нужно прогнать быстрее на занятии, уменьшите `max_users_per_stage`, `candidates_k` и `deep_epochs`.


In [27]:
CFG = SplitConfig(
    stage_days=14,
    min_history_days=60,
    positive_threshold=50.0,
    max_users_per_stage=10000,
    candidates_k=60,
    popular_fill_k=20,
    sequence_len=30,
)

DEEP_EPOCHS = 2
DEEP_BATCH_SIZE = 1024
DATA_ROOT = find_data_root(Path('.'))
DATA_ROOT

ACTIVE_USER_POOL = 1000
TOP_ITEM_POOL = 5000


## 2. Загрузка и препроцессинг KION

In [28]:
interactions, users, items = load_kion_data(DATA_ROOT)
interactions, users, items = preprocess_kion(interactions, users, items)

print('interactions:', interactions.shape)
print('users       :', users.shape)
print('items       :', items.shape)
print('date range  :', interactions['date'].min(), '->', interactions['date'].max())

interactions.head()


interactions: (5476251, 6)
users       : (840197, 5)
items       : (15963, 17)
date range  : 2021-03-13 00:00:00 -> 2021-08-22 00:00:00


,user_id,item_id,last_watch_dt,total_dur,watched_pct,date
0,176549,9506,2021-05-11,4250.0,72.0,2021-05-11
1,699317,1659,2021-05-29,8317.0,100.0,2021-05-29
2,656683,7107,2021-05-09,10.0,0.0,2021-05-09
3,864613,7638,2021-07-05,14483.0,100.0,2021-07-05
4,964868,9506,2021-04-30,6725.0,100.0,2021-04-30


## 2.1 Учебный сэмпл для быстрого EASE

Следуем идее легкого пайплайна из `Seminar5`: сначала ограничиваем user/item universe, а потом уже строим EASE-кандидатов и CTR-dataset.

`EASE` требует инверсию item-item матрицы, поэтому для ноутбука фиксируем:
- `10k` самых активных пользователей;
- не больше `5k` самых популярных айтемов внутри этого user pool.

Так ноутбук остается заметно легче, но при этом сохраняет реалистичный rerank pipeline.


In [29]:
active_users = interactions['user_id'].value_counts().head(ACTIVE_USER_POOL).index
interactions = interactions.loc[interactions['user_id'].isin(active_users)].copy()

top_items = interactions['item_id'].value_counts().head(TOP_ITEM_POOL).index
interactions = interactions.loc[interactions['item_id'].isin(top_items)].copy()

users = users.loc[users['user_id'].isin(interactions['user_id'].unique())].copy()
items = items.loc[items['item_id'].isin(interactions['item_id'].unique())].copy()

print('sampled interactions:', interactions.shape)
print('sampled users       :', users.shape)
print('sampled items       :', items.shape)


sampled interactions: (161347, 6)
sampled users       : (770, 5)
sampled items       : (5000, 17)


## 3. Stage-окна

Используем три непересекающихся окна:
- `train`: по истории до старта окна собираем кандидатов, а label берем из следующего 14-дневного target-окна;
- `val`: аналогично;
- `test`: аналогично.

Так мы честно эмулируем production-сценарий rerank-а.


In [30]:
windows = make_stage_windows(interactions, CFG)
for window in windows:
    print(window)


StageWindow(name='train', history_end=Timestamp('2021-07-11 00:00:00'), target_start=Timestamp('2021-07-12 00:00:00'), target_end=Timestamp('2021-07-25 00:00:00'))
StageWindow(name='val', history_end=Timestamp('2021-07-25 00:00:00'), target_start=Timestamp('2021-07-26 00:00:00'), target_end=Timestamp('2021-08-08 00:00:00'))
StageWindow(name='test', history_end=Timestamp('2021-08-08 00:00:00'), target_start=Timestamp('2021-08-09 00:00:00'), target_end=Timestamp('2021-08-22 00:00:00'))


## 4. Candidate generation через EASE и сбор CTR-датасета

На каждом stage мы:
1. обучаем `EASE` только на истории до `history_end`;
2. берем `top-k` кандидатов;
3. дополняем популярными айтемами;
4. ставим `label = 1`, если пользователь действительно посмотрел этот айтем в target-окне с `watched_pct >= positive_threshold`.


In [31]:
datasets = {}
for window in windows:
    stage_history = interactions.loc[interactions['date'] <= window.history_end].copy()
    ease = EASE(l2=500.0).fit(stage_history)
    stage_df = build_stage_dataset(interactions, users, items, ease, window, CFG)
    datasets[window.name] = stage_df
    print(window.name, stage_df.shape, 'positive rate =', round(stage_df['label'].mean(), 4))

train_df = datasets['train']
val_df = datasets['val']
test_df = datasets['test']


train (49140, 33) positive rate = 0.0264
val (49260, 33) positive rate = 0.0268
test (47280, 33) positive rate = 0.026


In [32]:
train_df.head()


,user_id,item_id,ease_score,ease_rank,label,content_type,first_genre,first_director,release_year,release_year_bucket,for_kids,age,income,sex,kids_flg,user_hist_interactions,user_hist_items,user_avg_watch_pct,user_days_since_last,item_hist_interactions,item_hist_users,item_avg_watch_pct,item_days_since_last,ui_hist_interactions,ui_avg_watch_pct,ui_days_since_last,ug_hist_interactions,ug_avg_watch_pct,ud_hist_interactions,ud_avg_watch_pct,history_items,history_len,stage
0,1000662,10464,0.286524,1,0,film,драмы,Чарли Бюхлер,2020,2011-2020,0,age_25_34,income_20_40,М,1.0,85,85,80.6,2,201,201,54.477612,0,0.0,0.0,0.0,17.0,83.058824,0.0,0.0,"[9169, 772, 869, 14709, 6353, 10073, 10647, 12...",85,train
1,1000662,12995,0.276446,2,0,film,боевики,Гуань Ху,2020,2011-2020,0,age_25_34,income_20_40,М,1.0,85,85,80.6,2,236,236,47.847458,0,0.0,0.0,0.0,26.0,83.153846,0.0,0.0,"[9169, 772, 869, 14709, 6353, 10073, 10647, 12...",85,train
2,1000662,10440,0.272278,3,0,series,триллеры,Душан Глигоров,2021,2021+,0,age_25_34,income_20_40,М,1.0,85,85,80.6,2,319,319,53.573668,0,0.0,0.0,0.0,3.0,95.333333,0.0,0.0,"[9169, 772, 869, 14709, 6353, 10073, 10647, 12...",85,train
3,1000662,7102,0.267784,4,0,film,боевики,Дэвид Хэкл,2019,2011-2020,0,age_25_34,income_20_40,М,1.0,85,85,80.6,2,207,207,65.217391,0,0.0,0.0,0.0,26.0,83.153846,1.0,100.0,"[9169, 772, 869, 14709, 6353, 10073, 10647, 12...",85,train
4,1000662,14431,0.266027,5,0,film,ужасы,Святослав Подгаевский,2021,2021+,0,age_25_34,income_20_40,М,1.0,85,85,80.6,2,203,203,62.201970,0,0.0,0.0,0.0,12.0,86.000000,0.0,0.0,"[9169, 772, 869, 14709, 6353, 10073, 10647, 12...",85,train


## 5. Baseline: качество EASE на top-10

Здесь `ease_score` уже можно воспринимать как baseline score на candidate set.


In [33]:
ease_val_metrics = evaluate_ranking(val_df, 'ease_score')
ease_test_metrics = evaluate_ranking(test_df, 'ease_score')

pd.DataFrame([
    {'split': 'val', **ease_val_metrics},
    {'split': 'test', **ease_test_metrics},
])


,split,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
0,val,0.196059,0.170257,0.321791,0.217575,0.527058,0.280869
1,test,0.236901,0.216974,0.364564,0.261351,0.525599,0.313526


## 6. LGBMClassifier vs LGBMRanker

Используем один и тот же candidate dataset, но разные objectives:
- `Classifier`: бинарная CTR/engagement классификация;
- `Ranker`: `lambdarank`, query = `user_id`.


In [34]:
train_lgbm, val_lgbm, test_lgbm, feature_cols, cat_cols = prepare_lgbm_matrices(train_df, val_df, test_df)
clf, ranker = fit_lgbm_models(train_lgbm, val_lgbm, feature_cols, cat_cols)

val_lgbm['lgbm_classifier_score'] = clf.predict_proba(val_lgbm[feature_cols])[:, 1]
test_lgbm['lgbm_classifier_score'] = clf.predict_proba(test_lgbm[feature_cols])[:, 1]

val_rank_input = val_lgbm.sort_values('user_id').copy()
test_rank_input = test_lgbm.sort_values('user_id').copy()
val_rank_input['lgbm_ranker_score'] = ranker.predict(val_rank_input[feature_cols])
test_rank_input['lgbm_ranker_score'] = ranker.predict(test_rank_input[feature_cols])

classical_metrics = pd.DataFrame([
    {'model': 'EASE candidate score', **evaluate_ranking(test_df, 'ease_score')},
    {'model': 'LGBMClassifier', **evaluate_ranking(test_lgbm, 'lgbm_classifier_score')},
    {'model': 'LGBMRanker', **evaluate_ranking(test_rank_input, 'lgbm_ranker_score')},
]).sort_values(['ndcg@10', 'recall@10'], ascending=False)

classical_metrics


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 1296, number of negative: 47844
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000848 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3260
[LightGBM] [Info] Number of data points in the train set: 49140, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.026374 -> initscore=-3.608663
[LightGBM] [Info] Start training from score -3.608663
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large num

,model,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
0,EASE candidate score,0.236901,0.216974,0.364564,0.261351,0.525599,0.313526
1,LGBMClassifier,0.247590,0.201597,0.342888,0.235126,0.528973,0.293902
2,LGBMRanker,0.230921,0.197378,0.337261,0.233333,0.514185,0.289999


## 7. Подготовка encoder-а для deep CTR

Для deep-моделей используем:
- категориальные признаки пользователя и айтема;
- числовые агрегаты из history;
- `history_items` для sequential model.


In [35]:
deep_cat_cols = [
    'age', 'income', 'sex', 'content_type', 'first_genre',
    'first_director', 'release_year_bucket', 'for_kids', 'kids_flg'
]
deep_num_cols = [c for c in feature_cols if c not in cat_cols]

encoder = FeatureEncoder(cat_cols=deep_cat_cols, num_cols=deep_num_cols).fit([train_df, val_df, test_df])
cat_cardinalities = [len(encoder.cat_maps[col]) + 1 for col in deep_cat_cols]
item_vocab_size = len(encoder.item_map) + 1

print('cat cardinalities:', dict(zip(deep_cat_cols, cat_cardinalities)))
print('item vocab size  :', item_vocab_size)
print('num features     :', len(deep_num_cols))


cat cardinalities: {'age': 8, 'income': 8, 'sex': 4, 'content_type': 3, 'first_genre': 36, 'first_director': 1375, 'release_year_bucket': 7, 'for_kids': 3, 'kids_flg': 4}
item vocab size  : 4939
num features     : 19


## 8. DCNv2 (lite)

Компактная PyTorch-реализация в стиле `cross network + deep tower`.


In [36]:
dcn_model = DCNv2Lite(cat_cardinalities=cat_cardinalities, num_dim=len(deep_num_cols))
dcn_model = train_torch_model(
    dcn_model,
    train_df=train_df,
    val_df=val_df,
    encoder=encoder,
    epochs=DEEP_EPOCHS,
    batch_size=DEEP_BATCH_SIZE,
    seq_len=CFG.sequence_len,
)

test_dcn = test_df.copy()
test_dcn['score'] = predict_torch_model(dcn_model, test_dcn, encoder, batch_size=2048, seq_len=CFG.sequence_len)
evaluate_ranking(test_dcn, 'score')


{'recall@5': 0.15207419731565633,
 'ndcg@5': 0.14418427825431523,
 'recall@10': 0.21563059268556314,
 'ndcg@10': 0.1671804244519902,
 'recall@20': 0.3273010543271268,
 'ndcg@20': 0.20201628119324705}

## 9. FinalNet (lite)

Seminar-friendly версия factorized/gated interaction блока, вдохновленная `FinalNet`.


In [37]:
finalnet_model = FinalNetLite(cat_cardinalities=cat_cardinalities, num_dim=len(deep_num_cols))
finalnet_model = train_torch_model(
    finalnet_model,
    train_df=train_df,
    val_df=val_df,
    encoder=encoder,
    epochs=DEEP_EPOCHS,
    batch_size=DEEP_BATCH_SIZE,
    seq_len=CFG.sequence_len,
)

test_finalnet = test_df.copy()
test_finalnet['score'] = predict_torch_model(finalnet_model, test_finalnet, encoder, batch_size=2048, seq_len=CFG.sequence_len)
evaluate_ranking(test_finalnet, 'score')


{'recall@5': 0.20700574648686643,
 'ndcg@5': 0.17438944450752975,
 'recall@10': 0.29959766843596797,
 'ndcg@10': 0.20789508673215537,
 'recall@20': 0.4661760699841871,
 'ndcg@20': 0.26198180850986985}

## 10. TransAct-style sequence model (lite)

Здесь уже используется пользовательская последовательность `history_items` и target-item embedding.


In [38]:
transact_model = TransActLite(
    cat_cardinalities=cat_cardinalities,
    item_vocab=item_vocab_size,
    num_dim=len(deep_num_cols),
    seq_len=CFG.sequence_len,
)
transact_model = train_torch_model(
    transact_model,
    train_df=train_df,
    val_df=val_df,
    encoder=encoder,
    epochs=DEEP_EPOCHS,
    batch_size=DEEP_BATCH_SIZE,
    seq_len=CFG.sequence_len,
)

test_transact = test_df.copy()
test_transact['score'] = predict_torch_model(transact_model, test_transact, encoder, batch_size=2048, seq_len=CFG.sequence_len)
evaluate_ranking(test_transact, 'score')


{'recall@5': 0.18056032751961343,
 'ndcg@5': 0.1483804664138077,
 'recall@10': 0.30787700433539117,
 'ndcg@10': 0.19600931718147646,
 'recall@20': 0.5038001310702312,
 'ndcg@20': 0.2569987853597323}

## 11. Финальный лидерборд по top-10

In [40]:
leaderboard = build_leaderboard({
    'EASE candidate score': test_df.rename(columns={'ease_score': 'score'}),
    'LGBMClassifier': test_lgbm.rename(columns={'lgbm_classifier_score': 'score'}),
    'LGBMRanker': test_rank_input.rename(columns={'lgbm_ranker_score': 'score'}),
    'DCNv2 (lite)': test_dcn,
    'FinalNet (lite)': test_finalnet,
    'TransAct-style (lite)': test_transact,
})
leaderboard


,model,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
0,EASE candidate score,0.236901,0.216974,0.364564,0.261351,0.525599,0.313526
1,LGBMClassifier,0.247590,0.201597,0.342888,0.235126,0.528973,0.293902
2,LGBMRanker,0.230921,0.197378,0.337261,0.233333,0.514185,0.289999
3,FinalNet (lite),0.207006,0.174389,0.299598,0.207895,0.466176,0.261982
4,TransAct-style (lite),0.180560,0.148380,0.307877,0.196009,0.503800,0.256999
5,DCNv2 (lite),0.152074,0.144184,0.215631,0.167180,0.327301,0.202016


## 12. Что можно улучшить дальше

1. Сделать candidate generation сильнее: `ItemKNN`, `ALS`, `EASE + popularity mix`, `two-tower`.
2. Добавить impression-level/context features, если доступны.
3. Усложнить target: отдельные классы `click / long_watch / completion`.
4. Вынести deep-модели на официальный `FuxiCTR/BARS` pipeline и повторить сравнение уже на их конфигурациях.
5. Для sequential части сравнить `TransAct` не только с tabular CTR, но и с `DIN/DIEN/BST`.
